# Task 2.2: Reproduction of Core Contribution

## Paper: "Kernel Methods for Deep Learning" — Cho & Saul (NIPS 2009)

---

### Contribution Being Reproduced

We are reproducing the **arc-cosine kernel** family (Section 2) and its use as a custom kernel inside an SVM classifier (Section 2.4). Specifically, we implement:

1. **Single-layer arc-cosine kernels** of degree n = 0, 1, and 2 (Equations 3–7).
2. **Multilayer (recursive) arc-cosine kernels** that compose the single-layer kernel multiple times to mimic deep threshold networks (Equations 12–13).

We then train SVMs with these kernels on the digits dataset prepared in Task 2.1 and evaluate classification accuracy.

### Evaluation Metric

**Classification accuracy** on the held-out test set, consistent with the paper's reporting of test error rates (Figures 2–5, Section 2.4).

In [1]:
# ============================================================
# SETUP: imports and hyperparameters
# ============================================================
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Reproducibility ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Data split ratios ---
TEST_SIZE = 0.3
VAL_SIZE = 0.15

# --- SVM hyperparameter (margin penalty) ---
# We will tune C using the validation set, following the paper's methodology (Section 2.4)
C_VALUES = [0.1, 1.0, 10.0, 100.0]

print(f"Random seed: {RANDOM_SEED}")
print(f"C values to search: {C_VALUES}")

Matplotlib is building the font cache; this may take a moment.


Random seed: 42
C values to search: [0.1, 1.0, 10.0, 100.0]


We set up all imports and hyperparameters at the top for reproducibility. The SVM penalty parameter C will be tuned via the validation set, which mirrors the methodology described in Section 2.4 of the paper — they used the last 2000 training examples as a validation set to choose C before retraining on the full training data.

In [2]:
# ============================================================
# LOAD AND PREPARE DATA (same as Task 2.1)
# ============================================================
digits = load_digits()
X, y = digits.data, digits.target

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SIZE, random_state=RANDOM_SEED, stratify=y_trainval
)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

Train: 1068, Val: 189, Test: 540


We reload and split the digits dataset exactly as in Task 2.1 to ensure consistency. The training, validation, and test sets are identical across all notebooks because we use the same random seed and split ratios.

In [3]:
# ============================================================
# IMPLEMENT THE ANGULAR DEPENDENCE FUNCTIONS J_n(theta)
# These are from Equations 5, 6, and 7 in the paper.
# ============================================================

def J_0(theta):
    """J_0(theta) = pi - theta   [Eq. 5]"""
    return np.pi - theta

def J_1(theta):
    """J_1(theta) = sin(theta) + (pi - theta) * cos(theta)   [Eq. 6]"""
    return np.sin(theta) + (np.pi - theta) * np.cos(theta)

def J_2(theta):
    """J_2(theta) = 3*sin(theta)*cos(theta) + (pi - theta)*(1 + 2*cos^2(theta))   [Eq. 7]"""
    return 3 * np.sin(theta) * np.cos(theta) + (np.pi - theta) * (1 + 2 * np.cos(theta)**2)

# Map degree to J function
J_FUNCTIONS = {0: J_0, 1: J_1, 2: J_2}

# Quick sanity check: J_0(0) should be pi, J_0(pi) should be 0
print(f"J_0(0) = {J_0(0):.4f}  (expected: {np.pi:.4f})")
print(f"J_0(pi) = {J_0(np.pi):.4f}  (expected: 0.0000)")
print(f"J_1(0) = {J_1(0):.4f}  (expected: {np.pi:.4f})")
print(f"J_1(pi/2) = {J_1(np.pi/2):.4f}  (expected: 1.0000)")

J_0(0) = 3.1416  (expected: 3.1416)
J_0(pi) = 0.0000  (expected: 0.0000)
J_1(0) = 3.1416  (expected: 3.1416)
J_1(pi/2) = 1.0000  (expected: 1.0000)


These three functions implement the angular dependence $J_n(\theta)$ from Equations 5, 6, and 7 of the paper. They capture the interesting part of the kernel — how similarity varies with the angle between inputs. The magnitude dependence is trivial (just $\|x\|^n \|y\|^n$), but the angular dependence is what makes these kernels behave differently from standard polynomial or RBF kernels. We verify with simple sanity checks: for instance, when two vectors are perfectly aligned ($\theta = 0$), $J_0(0) = \pi$, confirming maximum similarity.

In [4]:
# ============================================================
# IMPLEMENT THE SINGLE-LAYER ARC-COSINE KERNEL (Eq. 3)
# k_n(x, y) = (1/pi) * ||x||^n * ||y||^n * J_n(theta)
# where theta = arccos( (x . y) / (||x|| * ||y||) )    [Eq. 2]
# ============================================================

def arc_cosine_kernel_single(X, Y, n=1):
    """
    Compute the single-layer arc-cosine kernel matrix between X and Y.
    
    Parameters:
        X: array of shape (n_samples_X, n_features)
        Y: array of shape (n_samples_Y, n_features)
        n: degree of the kernel (0, 1, or 2)
    
    Returns:
        K: kernel matrix of shape (n_samples_X, n_samples_Y)
    """
    J_n = J_FUNCTIONS[n]
    
    # Compute norms of each sample
    norms_X = np.linalg.norm(X, axis=1)  # shape: (n_X,)
    norms_Y = np.linalg.norm(Y, axis=1)  # shape: (n_Y,)
    
    # Compute dot products: X @ Y^T gives (n_X, n_Y) matrix
    dot_products = X @ Y.T
    
    # Compute the angle theta between each pair (Eq. 2)
    # We need to clip the cosine values to [-1, 1] to avoid numerical issues with arccos
    norms_outer = np.outer(norms_X, norms_Y)
    
    # Handle zero-norm vectors: if either vector has zero norm, set cos_theta to 0
    cos_theta = np.where(norms_outer > 0, dot_products / norms_outer, 0.0)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    theta = np.arccos(cos_theta)  # Eq. 2
    
    # Compute angular part: J_n(theta)
    angular_part = J_n(theta)
    
    # Compute magnitude part: ||x||^n * ||y||^n
    magnitude_part = np.outer(norms_X**n, norms_Y**n)
    
    # Full kernel: k_n(x,y) = (1/pi) * ||x||^n * ||y||^n * J_n(theta)   [Eq. 3]
    K = (1.0 / np.pi) * magnitude_part * angular_part
    
    return K

# Quick test: k_0(x, x) should be 1 for any x (maps to unit hypersphere)
test_x = X_train[:3]
K_test = arc_cosine_kernel_single(test_x, test_x, n=0)
print(f"k_0(x, x) diagonal (should be ~1.0): {np.diag(K_test)}")

# k_1(x, x) should equal ||x||^2
K_test_1 = arc_cosine_kernel_single(test_x, test_x, n=1)
expected_norms_sq = np.sum(test_x**2, axis=1)
print(f"k_1(x,x) diagonal: {np.diag(K_test_1)}")
print(f"||x||^2 values   : {expected_norms_sq}")
print(f"Match: {np.allclose(np.diag(K_test_1), expected_norms_sq)}")

k_0(x, x) diagonal (should be ~1.0): [1.         1.         0.99999999]
k_1(x,x) diagonal: [15.37809028 16.14243941 17.92011196]
||x||^2 values   : [15.37809028 16.14243941 17.92011196]
Match: True


This is the core implementation of the **single-layer arc-cosine kernel** from Equation 3. For each pair of inputs $(x, y)$, it computes the angle $\theta$ between them (Eq. 2), evaluates the angular function $J_n(\theta)$, and multiplies by the norm factor $\|x\|^n \|y\|^n$. We verify two properties the paper highlights in Section 2.1: (i) the $n=0$ kernel maps everything onto a unit hypersphere so $k_0(x,x) = 1$, and (ii) the $n=1$ kernel preserves input norms so $k_1(x,x) = \|x\|^2$. Both checks pass, which gives us confidence the implementation is correct.

In [5]:
# ============================================================
# IMPLEMENT THE MULTILAYER ARC-COSINE KERNEL (Eqs. 12-13)
#
# Recursion:
#   Base case (l=1): k_n^(1)(x,y) = single-layer kernel from Eq. 3
#   Inductive step:  k_n^(l+1)(x,y) = (1/pi) * [k_n^(l)(x,x) * k_n^(l)(y,y)]^(n/2) * J_n(theta_n^(l))
#   where theta_n^(l) = arccos( k_n^(l)(x,y) / sqrt(k_n^(l)(x,x) * k_n^(l)(y,y)) )
#
# Following the paper's empirical finding (Section 2.4):
#   - Use the specified degree n at layer 1
#   - Use n=1 at all subsequent layers (l > 1)
# ============================================================

def arc_cosine_kernel_multilayer(X, Y, n=1, depth=1):
    """
    Compute the multilayer arc-cosine kernel.
    
    At layer 1, uses degree n.
    At layers 2+, uses degree 1 (norm-preserving), as recommended
    by the paper's experiments (Section 2.4).
    
    Parameters:
        X: array of shape (n_samples_X, n_features)
        Y: array of shape (n_samples_Y, n_features)
        n: degree for the first layer
        depth: number of recursive layers (l)
    
    Returns:
        K: kernel matrix of shape (n_samples_X, n_samples_Y)
    """
    # Base case: single-layer kernel with degree n
    K = arc_cosine_kernel_single(X, Y, n=n)
    
    if depth == 1:
        return K
    
    # For layers 2 through depth, apply recursive composition with degree 1
    # We also need the diagonal entries k(x,x) and k(y,y)
    K_xx_diag = np.diag(arc_cosine_kernel_single(X, X, n=n))
    K_yy_diag = np.diag(arc_cosine_kernel_single(Y, Y, n=n))
    
    for l in range(2, depth + 1):
        # Use n=1 at layers > 1
        layer_n = 1
        J_n = J_FUNCTIONS[layer_n]
        
        # Compute theta^(l-1) from Eq. 13
        # theta = arccos( K(x,y) / sqrt(K(x,x) * K(y,y)) )
        denom = np.sqrt(np.outer(K_xx_diag, K_yy_diag))
        cos_theta = np.where(denom > 0, K / denom, 0.0)
        cos_theta = np.clip(cos_theta, -1.0, 1.0)
        theta = np.arccos(cos_theta)  # Eq. 13
        
        # Compute new kernel from Eq. 12
        # k^(l+1) = (1/pi) * [k^(l)(x,x) * k^(l)(y,y)]^(n/2) * J_n(theta)
        magnitude_part = np.outer(K_xx_diag, K_yy_diag) ** (layer_n / 2.0)
        K_new = (1.0 / np.pi) * magnitude_part * J_n(theta)
        
        # Update diagonals for next iteration
        K_xx_diag = (1.0 / np.pi) * (K_xx_diag ** layer_n) * J_n(np.zeros(len(K_xx_diag)))
        K_yy_diag = (1.0 / np.pi) * (K_yy_diag ** layer_n) * J_n(np.zeros(len(K_yy_diag)))
        
        K = K_new
    
    return K

# Quick test: multilayer with depth=1 should equal single-layer
K_single = arc_cosine_kernel_single(test_x, test_x, n=1)
K_multi_1 = arc_cosine_kernel_multilayer(test_x, test_x, n=1, depth=1)
print(f"Single-layer == Multilayer(depth=1): {np.allclose(K_single, K_multi_1)}")

# Test depth=2
K_multi_2 = arc_cosine_kernel_multilayer(test_x, test_x, n=1, depth=2)
print(f"Depth=2 kernel diagonal: {np.diag(K_multi_2)}")
print(f"Depth=2 kernel shape: {K_multi_2.shape}")

Single-layer == Multilayer(depth=1): True
Depth=2 kernel diagonal: [15.37809028 16.14243941 17.92011196]
Depth=2 kernel shape: (3, 3)


This implements the **multilayer (recursive) arc-cosine kernel** from Equations 12 and 13. Starting from the single-layer kernel as the base case, each subsequent layer computes a new kernel by: (a) computing the angle $\theta^{(\ell)}_n$ in the current feature space (Eq. 13), and (b) applying the kernel formula again (Eq. 12). Following the paper's key empirical finding in Section 2.4, we always use degree $n=1$ at layers $\ell > 1$ because only the $n=1$ kernel preserves input norms across layers — using $n=0$ would collapse everything to a unit sphere, and $n \geq 2$ would cause the dynamic range to explode. We verify that depth=1 gives the same result as the single-layer kernel.

In [6]:
# ============================================================
# TRAIN SVM WITH ARC-COSINE KERNEL: Tune C on validation set
# Following Section 2.4 methodology
# ============================================================

def train_and_evaluate_svm(X_train, y_train, X_val, y_val, X_test, y_test,
                           kernel_fn, kernel_name="arc-cosine"):
    """
    Train SVM with a precomputed kernel matrix.
    Tune C on validation set, retrain on train+val, and evaluate on test.
    """
    # Step 1: compute kernel matrices
    K_train = kernel_fn(X_train, X_train)
    K_val   = kernel_fn(X_val, X_train)
    
    # Step 2: tune C on validation set
    best_c, best_val_acc = None, 0
    for c in C_VALUES:
        svm = SVC(kernel='precomputed', C=c, random_state=RANDOM_SEED)
        svm.fit(K_train, y_train)
        val_acc = accuracy_score(y_val, svm.predict(K_val))
        if val_acc > best_val_acc:
            best_c, best_val_acc = c, val_acc
    
    print(f"  [{kernel_name}] Best C = {best_c} (val accuracy = {best_val_acc:.4f})")
    
    # Step 3: retrain on train+val combined (following paper's methodology)
    X_full = np.vstack([X_train, X_val])
    y_full = np.concatenate([y_train, y_val])
    K_full = kernel_fn(X_full, X_full)
    K_test_full = kernel_fn(X_test, X_full)
    
    svm_final = SVC(kernel='precomputed', C=best_c, random_state=RANDOM_SEED)
    svm_final.fit(K_full, y_full)
    
    # Step 4: evaluate on test set
    y_pred = svm_final.predict(K_test_full)
    test_acc = accuracy_score(y_test, y_pred)
    test_err = 1.0 - test_acc
    
    print(f"  [{kernel_name}] Test accuracy = {test_acc:.4f}  (error rate = {test_err:.4f})")
    
    return test_acc, test_err, best_c, y_pred

print("SVM training pipeline ready.")

SVM training pipeline ready.


This function trains an SVM with a precomputed kernel matrix and tunes the margin penalty parameter C on the validation set, exactly mirroring the experimental methodology from Section 2.4 of the paper. The authors used libSVM and tuned C by holding out the last 2000 training examples for validation, then retrained using all training data after selecting the best C. We follow the same protocol here with scikit-learn's `SVC(kernel='precomputed')`.

In [7]:
# ============================================================
# EXPERIMENT 1: Single-layer arc-cosine kernels (n=0, 1, 2)
# Corresponds to Section 2.4, single recursion depth (l=1)
# ============================================================

print("=" * 60)
print("EXPERIMENT 1: Single-layer arc-cosine kernels (depth = 1)")
print("=" * 60)

results = {}

for degree in [0, 1, 2]:
    degree_names = {0: 'Step (n=0)', 1: 'Ramp (n=1)', 2: 'Quarter-pipe (n=2)'}
    name = degree_names[degree]
    print(f"\nTraining SVM with {name} kernel, depth=1...")
    
    kernel_fn = lambda X, Y, n=degree: arc_cosine_kernel_single(X, Y, n=n)
    acc, err, best_c, y_pred = train_and_evaluate_svm(
        X_train, y_train, X_val, y_val, X_test, y_test,
        kernel_fn, kernel_name=name
    )
    results[f"{name}, l=1"] = {'accuracy': acc, 'error': err, 'C': best_c, 'y_pred': y_pred}

print("\n" + "=" * 60)
print("Single-layer results summary:")
for name, res in results.items():
    print(f"  {name}: accuracy = {res['accuracy']:.4f}, error = {res['error']:.4f}")

EXPERIMENT 1: Single-layer arc-cosine kernels (depth = 1)

Training SVM with Step (n=0) kernel, depth=1...
  [Step (n=0)] Best C = 10.0 (val accuracy = 0.9894)
  [Step (n=0)] Test accuracy = 0.9907  (error rate = 0.0093)

Training SVM with Ramp (n=1) kernel, depth=1...


  [Ramp (n=1)] Best C = 1.0 (val accuracy = 0.9947)


  [Ramp (n=1)] Test accuracy = 0.9907  (error rate = 0.0093)

Training SVM with Quarter-pipe (n=2) kernel, depth=1...
  [Quarter-pipe (n=2)] Best C = 0.1 (val accuracy = 0.9947)


  [Quarter-pipe (n=2)] Test accuracy = 0.9852  (error rate = 0.0148)

Single-layer results summary:
  Step (n=0), l=1: accuracy = 0.9907, error = 0.0093
  Ramp (n=1), l=1: accuracy = 0.9907, error = 0.0093
  Quarter-pipe (n=2), l=1: accuracy = 0.9852, error = 0.0148


We evaluate all three single-layer arc-cosine kernels corresponding to the three activation functions shown in Figure 1 of the paper: **Step** ($n=0$, like an array of perceptrons), **Ramp** ($n=1$, like ReLU — a piecewise linear mapping), and **Quarter-pipe** ($n=2$). In the paper (Figure 2), single-layer arc-cosine kernels already often outperform SVMs with RBF kernels and sometimes match deep belief nets on the rectangles-image and convex datasets.

In [8]:
# ============================================================
# EXPERIMENT 2: Multilayer arc-cosine kernels (depth 1-4)
# Corresponds to Section 2.3–2.4 and Figures 2-3
# Using n=1 at first layer (Ramp) with recursive depth
# ============================================================

print("=" * 60)
print("EXPERIMENT 2: Multilayer arc-cosine kernels (n=1 at layer 1)")
print("=" * 60)

multilayer_results = {}

for depth in [1, 2, 3, 4]:
    name = f"Ramp (n=1), depth={depth}"
    print(f"\nTraining SVM with n=1 kernel, depth={depth}...")
    
    kernel_fn = lambda X, Y, d=depth: arc_cosine_kernel_multilayer(X, Y, n=1, depth=d)
    acc, err, best_c, y_pred = train_and_evaluate_svm(
        X_train, y_train, X_val, y_val, X_test, y_test,
        kernel_fn, kernel_name=name
    )
    multilayer_results[depth] = {'accuracy': acc, 'error': err, 'C': best_c}

print("\n" + "=" * 60)
print("Multilayer results summary (n=1 at all layers):")
for depth, res in multilayer_results.items():
    print(f"  depth={depth}: accuracy = {res['accuracy']:.4f}, error = {res['error']:.4f}")

EXPERIMENT 2: Multilayer arc-cosine kernels (n=1 at layer 1)

Training SVM with n=1 kernel, depth=1...


  [Ramp (n=1), depth=1] Best C = 1.0 (val accuracy = 0.9947)
  [Ramp (n=1), depth=1] Test accuracy = 0.9907  (error rate = 0.0093)

Training SVM with n=1 kernel, depth=2...


  [Ramp (n=1), depth=2] Best C = 1.0 (val accuracy = 0.9947)


  [Ramp (n=1), depth=2] Test accuracy = 0.9907  (error rate = 0.0093)

Training SVM with n=1 kernel, depth=3...


  [Ramp (n=1), depth=3] Best C = 1.0 (val accuracy = 0.9947)


  [Ramp (n=1), depth=3] Test accuracy = 0.9907  (error rate = 0.0093)

Training SVM with n=1 kernel, depth=4...


  [Ramp (n=1), depth=4] Best C = 1.0 (val accuracy = 0.9947)


  [Ramp (n=1), depth=4] Test accuracy = 0.9907  (error rate = 0.0093)

Multilayer results summary (n=1 at all layers):
  depth=1: accuracy = 0.9907, error = 0.0093
  depth=2: accuracy = 0.9907, error = 0.0093
  depth=3: accuracy = 0.9907, error = 0.0093
  depth=4: accuracy = 0.9907, error = 0.0093


This experiment evaluates the **multilayer arc-cosine kernel** at varying depths, implementing the recursive composition from Equations 12–13. The paper showed (Figures 2–3) that multilayer kernels often outperform their single-layer counterparts, suggesting that the deeper kernels capture some of the benefits typically associated with deep architectures. We use $n=1$ (Ramp) at all layers, following the paper's empirical finding that only $n=1$ works well at higher recursion depths because it preserves input norms (Section 2.4).

In [9]:
# ============================================================
# EXPERIMENT 3: Mixed-degree multilayer kernels
# Use n=0 at layer 1, n=1 at layers 2+  (as in Figures 2-3)
# ============================================================

print("=" * 60)
print("EXPERIMENT 3: Mixed-degree multilayer (n=0 at layer 1, n=1 at layers 2+)")
print("=" * 60)

mixed_results = {}

for depth in [1, 2, 3]:
    name = f"Step+Ramp, depth={depth}"
    print(f"\nTraining SVM with n=0 first layer, depth={depth}...")
    
    kernel_fn = lambda X, Y, d=depth: arc_cosine_kernel_multilayer(X, Y, n=0, depth=d)
    acc, err, best_c, y_pred = train_and_evaluate_svm(
        X_train, y_train, X_val, y_val, X_test, y_test,
        kernel_fn, kernel_name=name
    )
    mixed_results[depth] = {'accuracy': acc, 'error': err, 'C': best_c}

print("\n" + "=" * 60)
print("Mixed-degree results summary:")
for depth, res in mixed_results.items():
    print(f"  depth={depth}: accuracy = {res['accuracy']:.4f}, error = {res['error']:.4f}")

EXPERIMENT 3: Mixed-degree multilayer (n=0 at layer 1, n=1 at layers 2+)

Training SVM with n=0 first layer, depth=1...
  [Step+Ramp, depth=1] Best C = 10.0 (val accuracy = 0.9894)
  [Step+Ramp, depth=1] Test accuracy = 0.9907  (error rate = 0.0093)

Training SVM with n=0 first layer, depth=2...


  [Step+Ramp, depth=2] Best C = 10.0 (val accuracy = 0.9894)
  [Step+Ramp, depth=2] Test accuracy = 0.9907  (error rate = 0.0093)

Training SVM with n=0 first layer, depth=3...


  [Step+Ramp, depth=3] Best C = 10.0 (val accuracy = 0.9894)


  [Step+Ramp, depth=3] Test accuracy = 0.9907  (error rate = 0.0093)

Mixed-degree results summary:
  depth=1: accuracy = 0.9907, error = 0.0093
  depth=2: accuracy = 0.9907, error = 0.0093
  depth=3: accuracy = 0.9907, error = 0.0093


In this experiment we test the **mixed-degree multilayer kernel** configuration that the paper actually uses for its bar charts in Figures 2 and 3: the first layer uses $n=0$ (step function, corresponding to an array of perceptrons), while all subsequent layers use $n=1$ (ramp). This is exactly the setup described in Section 2.4 — each group of bars shows the test error when a particular kernel degree was used at the first layer, while $n=1$ was used at successive layers.

In [10]:
# ============================================================
# BASELINE: Standard SVM with RBF kernel for comparison
# The paper compares against SVM-RBF throughout (Figures 2-5)
# ============================================================

print("=" * 60)
print("BASELINE: SVM with RBF kernel")
print("=" * 60)

best_c_rbf, best_val_acc_rbf = None, 0
gamma_values = ['scale', 0.01, 0.1]
best_gamma = None

for gamma in gamma_values:
    for c in C_VALUES:
        svm_rbf = SVC(kernel='rbf', C=c, gamma=gamma, random_state=RANDOM_SEED)
        svm_rbf.fit(X_train, y_train)
        val_acc = accuracy_score(y_val, svm_rbf.predict(X_val))
        if val_acc > best_val_acc_rbf:
            best_c_rbf, best_val_acc_rbf, best_gamma = c, val_acc, gamma

# Retrain on full train+val
X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])
svm_rbf_final = SVC(kernel='rbf', C=best_c_rbf, gamma=best_gamma, random_state=RANDOM_SEED)
svm_rbf_final.fit(X_full, y_full)
y_pred_rbf = svm_rbf_final.predict(X_test)
rbf_acc = accuracy_score(y_test, y_pred_rbf)
rbf_err = 1.0 - rbf_acc

print(f"  Best C = {best_c_rbf}, gamma = {best_gamma}")
print(f"  RBF SVM test accuracy = {rbf_acc:.4f}  (error rate = {rbf_err:.4f})")

BASELINE: SVM with RBF kernel


  Best C = 10.0, gamma = scale
  RBF SVM test accuracy = 0.9907  (error rate = 0.0093)


We train a standard SVM with an RBF (Gaussian) kernel as a baseline, since the paper's primary comparison throughout is SVM-RBF (shown as a horizontal reference line in Figures 2–5). Unlike the arc-cosine kernel which has no continuous tuning parameters, the RBF kernel requires tuning the kernel width $\gamma$ — the paper explicitly notes this as a disadvantage of RBF kernels (Section 2.4). We search over both $C$ and $\gamma$ on the validation set.

In [11]:
# ============================================================
# SUMMARY TABLE: All results
# ============================================================

print("\n" + "=" * 70)
print(f'{"Method":<40} {"Accuracy":>10} {"Error Rate":>12}')
print("=" * 70)

# Single-layer results
for name, res in results.items():
    print(f'{name:<40} {res["accuracy"]:>10.4f} {res["error"]:>12.4f}')

print("-" * 70)

# Multilayer results (n=1 throughout)
for depth, res in multilayer_results.items():
    name = f'Ramp (n=1), depth={depth}'
    print(f'{name:<40} {res["accuracy"]:>10.4f} {res["error"]:>12.4f}')

print("-" * 70)

# Mixed-degree results
for depth, res in mixed_results.items():
    name = f'Step+Ramp (n=0→1), depth={depth}'
    print(f'{name:<40} {res["accuracy"]:>10.4f} {res["error"]:>12.4f}')

print("-" * 70)

# Baseline
print(f'{"SVM-RBF (baseline)":<40} {rbf_acc:>10.4f} {rbf_err:>12.4f}')
print("=" * 70)


Method                                     Accuracy   Error Rate
Step (n=0), l=1                              0.9907       0.0093
Ramp (n=1), l=1                              0.9907       0.0093
Quarter-pipe (n=2), l=1                      0.9852       0.0148
----------------------------------------------------------------------
Ramp (n=1), depth=1                          0.9907       0.0093
Ramp (n=1), depth=2                          0.9907       0.0093
Ramp (n=1), depth=3                          0.9907       0.0093
Ramp (n=1), depth=4                          0.9907       0.0093
----------------------------------------------------------------------
Step+Ramp (n=0→1), depth=1                   0.9907       0.0093
Step+Ramp (n=0→1), depth=2                   0.9907       0.0093
Step+Ramp (n=0→1), depth=3                   0.9907       0.0093
----------------------------------------------------------------------
SVM-RBF (baseline)                           0.9907       0.0093


The summary table above consolidates all our experimental results in one place. This lets us directly compare single-layer arc-cosine kernels, multilayer arc-cosine kernels at varying depths, and the RBF baseline — mirroring the structure of the bar charts in Figures 2 and 3 of the paper. The results on our toy digits dataset are discussed and interpreted in detail in Task 2.3.